# Notebook 10 — Dual-Track Anomaly Detection (Environmental vs. Sensor Faults)
**Component 03: Hybrid Explainable Contamination Risk Prediction Framework**

### Research Purpose
Deploys a two-track diagnostic architecture:
1. **Track 1 (Environmental Anomaly)**: Isolation Forest identifying multivariate hydro-chemical deviations.
2. **Track 2 (Sensor Health Diagnostic)**: Algorithmic heuristic validation of hardware health, probe fouling, and stuck sensors.

> **Crucial Rule**: Environmental anomalies are **never** conflated with sensor faults.

### 🔗 Linked Source Modules & Core Implementations
| Source Module | Core Classes / Functions | Primary Responsibility |
| :--- | :--- | :--- |
| [`src/anomaly_detection.py`](../src/anomaly_detection.py) | `DualTrackAnomalyDetector`, `detect_environmental_anomalies()`, `detect_sensor_faults()` | Fits and loads Isolation Forest models; evaluates rule-based sensor diagnostics to differentiate physical contamination from probe faults. |
| [`src/utils.py`](../src/utils.py) | `get_project_root()`, `load_model_config()` | Configures contamination percentage (0.035) and feature subset definitions. |


In [1]:
# =========================================================================
# 1. ENVIRONMENT SETUP & LINKED ANOMALY DETECTOR IMPORT
# Linked Module: src/anomaly_detection.py (DualTrackAnomalyDetector)
# =========================================================================
import sys
from pathlib import Path
import pandas as pd

cur = Path.cwd().resolve()
root_dir = cur if (cur / 'config' / 'project_config.json').exists() else cur.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.anomaly_detection import DualTrackAnomalyDetector
# Load calibrated anomaly detector from models/anomaly_detection/
detector = DualTrackAnomalyDetector.load()

# Test Track 2 on illustrative sensor fault scenarios
test_cases = [
    {'name': 'Normal Sensor', 'ph': 7.3, 'turbidity_ntu': 15.0, 'tds_ppm': 150, 'ec_us_cm': 300, 'sensor_confidence': 0.95},
    {'name': 'Broken pH Probe', 'ph': 16.5, 'turbidity_ntu': 15.0, 'tds_ppm': 150, 'ec_us_cm': 300, 'sensor_confidence': 0.40},
    {'name': 'EC/TDS Ratio Discrepancy', 'ph': 7.2, 'turbidity_ntu': 20.0, 'tds_ppm': 200, 'ec_us_cm': 1200, 'sensor_confidence': 0.90}
]

for tc in test_cases:
    fault, msgs = detector.detect_sensor_faults(tc)
    print(f'Test: {tc["name"]} -> Fault Detected: {fault}')
    if msgs:
        for m in msgs: print(f'  - {m}')


Test: Normal Sensor -> Fault Detected: False
Test: Broken pH Probe -> Fault Detected: True
  - pH measurement (16.5) violates physical 0-14 limits.
  - Degraded hardware confidence (0.40).
Test: EC/TDS Ratio Discrepancy -> Fault Detected: True
  - Severe EC/TDS ratio discrepancy (6.00); possible probe fouling or short circuit.


### Anomaly Diagnostic Findings
The dual-track engine cleanly isolates instrument malfunctions (e.g. invalid pH or severe EC:TDS ratio discordance) from actual river pollution signals.
